<div style="background:linear-gradient(135deg,#51247a,#7a3fa0);padding:24px 28px;border-radius:10px;margin-bottom:.8rem;border-bottom:4px solid #f0a500;"><div style="font-size:2.4rem;margin-bottom:6px;">🔗</div><div style="color:white;font-size:1.5rem;font-weight:700;">Collocation Analyser</div><div style="color:rgba(255,255,255,.82);font-size:.92rem;margin-top:4px;">Measure how strongly words attract each other in your corpus<br><a href="https://ladal.edu.au/tutorials/collocations/collocations.html" target="_blank" style="color:#f0c060;font-size:.85rem;">&#x2192; Open the full tutorial</a></div></div>


<div style="background:#eafaf1;border-left:4px solid #27ae60;border-radius:6px;padding:14px 18px;margin:.8rem 0;font-family:sans-serif;font-size:.9rem;"><b style="color:#1a6b3a;">&#x2705; How to use this tool:</b><ol style="margin:.6rem 0 0 0;padding-left:1.3em;line-height:2.0;"><li>Upload your <code>.txt</code> files to <b>MyTexts</b> (Step 2). Demo texts are loaded automatically if the folder is empty.</li><li>Edit the <b>node word</b> and settings in the configuration cell (Step 2).</li><li>Click <b>Kernel &#x2192; Restart &amp; Run All</b> to run the analysis.</li><li>Download results from <b>MyOutput</b> (Step 4).</li></ol></div>


<div style="background:#fff8e1;border-left:4px solid #f0a500;border-radius:6px;padding:14px 18px;margin:.8rem 0;font-family:sans-serif;font-size:.9rem;"><b style="color:#7a5000;">&#x26A1; Quick start:</b> Upload your files to the appropriate folder, edit the configuration cell, then click <b>Kernel &#x2192; Restart &amp; Run All</b>. The notebook runs from top to bottom automatically.</div>


<div style="background:#51247a;color:white;padding:10px 18px;border-radius:6px;margin:1.4rem 0 .4rem 0;font-family:sans-serif;"><span style="font-size:.75rem;opacity:.7;text-transform:uppercase;letter-spacing:.05em;">Step 1</span><br><b style="font-size:.98rem;">&#x2699;&#xFE0F; Set up the environment</b></div>


<div style="background:#f5f5f5;border-left:3px solid #bbb;border-radius:4px;padding:5px 12px;margin-bottom:3px;font-family:sans-serif;font-size:.78rem;color:#888;">&#x1F512; <em>Runs automatically — do not edit</em></div>


In [ ]:
# Step 1 — Load shared helpers (do not edit)
source("../helpers/ladal_helpers.R")


<div style="background:#51247a;color:white;padding:10px 18px;border-radius:6px;margin:1.4rem 0 .4rem 0;font-family:sans-serif;"><span style="font-size:.75rem;opacity:.7;text-transform:uppercase;letter-spacing:.05em;">Step 2</span><br><b style="font-size:.98rem;">&#x1F4C2; Upload your texts &amp; configure</b></div>


<div style="background:#f4f0f8;border:2px dashed #51247a;border-radius:8px;padding:18px 22px;margin:.6rem 0;font-family:sans-serif;"><b style="color:#51247a;font-size:.95rem;">&#x1F4C2; Upload your files to <code>MyTexts</code></b><ol style="margin:.6rem 0 0 0;padding-left:1.3em;font-size:.9rem;line-height:1.9;"><li>Open the <b>file browser panel</b> on the left (click the folder icon if hidden).</li><li>Double-click the <b><code>MyTexts</code></b> folder.</li><li><b>Drag and drop</b> your <code>.txt</code> files into that folder.</li><li>Then click <b>Kernel &#x2192; Restart &amp; Run All</b>.</li></ol><p style="margin:.5rem 0 0 0;font-size:.82rem;color:#888;">Only <code>.txt</code> files are accepted. If the folder is empty, demo data will be loaded automatically.</p></div>


<div style="background:#e8f4fd;border-left:4px solid #4085C6;border-radius:4px;padding:7px 14px;margin-bottom:3px;font-family:sans-serif;font-size:.82rem;color:#2a5ea8;">&#x270F;&#xFE0F; <b>Edit the values below</b>, then run this cell (Shift+Enter or click &#x25B6; Run in the toolbar).</div>


In [ ]:
# ── CONFIGURATION ──────────────────────────────────────────────────
# Edit the values below, then run this cell (Shift+Enter).

node_word        <- "language"  # word to find collocates of; "" = all collocations
window_size      <- 5           # context window in words (1–15)
min_freq         <- 2           # minimum co-occurrence count to include
#   Note: raising min_freq reduces noise from rare word pairs.
remove_stopwords <- TRUE        # remove function words before analysis
top_n            <- 20          # number of collocates to show in the plot
plot_measure     <- "MI"        # association measure to plot: "MI", "t_score", "log_ratio"


<div style="background:#51247a;color:white;padding:10px 18px;border-radius:6px;margin:1.4rem 0 .4rem 0;font-family:sans-serif;"><span style="font-size:.75rem;opacity:.7;text-transform:uppercase;letter-spacing:.05em;">Step 3</span><br><b style="font-size:.98rem;">&#x1F4CA; Run the analysis</b></div>


<div style="background:#f5f5f5;border-left:3px solid #bbb;border-radius:4px;padding:5px 12px;margin-bottom:3px;font-family:sans-serif;font-size:.78rem;color:#888;">&#x1F512; <em>Runs automatically — do not edit</em></div>


In [ ]:
suppressPackageStartupMessages(library(quanteda))
suppressPackageStartupMessages(library(ggplot2))
suppressPackageStartupMessages(library(dplyr))
suppressPackageStartupMessages(library(tidyr))

seed_demo_texts("MyTexts")

texts <- tryCatch(load_texts("MyTexts"),
  error = function(e) { .err(conditionMessage(e)); NULL })

if (!is.null(texts)) {
  show_corpus_summary(texts)
  valid_measures <- c("MI", "t_score", "log_ratio")
  if (!plot_measure %in% valid_measures) {
    .warn(paste0("plot_measure \"", plot_measure, "\" not recognised. ",
                 "Using \"MI\". Valid options: ", paste(valid_measures, collapse=", ")))
    plot_measure <- "MI"
  }
  .prog("Building co-occurrence matrix...", 25)
  toks <- tokens(texts, remove_punct = TRUE, remove_numbers = TRUE) |>
    tokens_tolower()
  if (remove_stopwords)
    toks <- tokens_remove(toks, stopwords("en"), padding = FALSE)
  fcm_obj <- fcm(toks, context = "window",
                 window = window_size, ordered = FALSE, tri = FALSE)
  node <- tolower(trimws(node_word))
  if (nzchar(node)) {
    if (!node %in% featnames(fcm_obj))
      stop("Word \"", node, "\" not found in corpus. ",
           "Check spelling, or set remove_stopwords=FALSE if it is a function word.")
    fcm_obj <- fcm_select(fcm_obj, pattern = node, valuetype = "fixed")
  }
  .prog("Calculating association measures...", 60)
  wf  <- colSums(dfm(toks)); N <- sum(wf)
  fcm_df <- as.data.frame(as.matrix(fcm_obj)) |>
    tibble::rownames_to_column("w1") |>
    tidyr::pivot_longer(-w1, names_to = "w2", values_to = "O") |>
    dplyr::filter(O >= min_freq, w1 != w2) |>
    dplyr::mutate(
      f1 = as.integer(wf[w1]), f2 = as.integer(wf[w2]),
      E  = pmax((f1 * f2) / N, .Machine$double.eps),
      MI        = log2(O / E),
      t_score   = (O - E) / sqrt(pmax(O, 1)),
      log_ratio = log2((O / N) / (E / N))
    ) |>
    dplyr::select(w1, w2, O, f1, f2, MI, t_score, log_ratio) |>
    dplyr::arrange(dplyr::desc(.data[[plot_measure]]))
  .prog("Plotting...", 85)
  measure_labels <- c(MI="Mutual Information (MI)",
                      t_score="t-score", log_ratio="Log Ratio")
  y_label <- measure_labels[[plot_measure]]
  title_str <- if (nzchar(node))
    paste0("Top ", top_n, ' collocates of "', node, '" by ', plot_measure)
  else paste("Top", top_n, "collocations by", plot_measure)
  p <- ggplot(head(fcm_df, top_n),
              aes(x = reorder(w2, .data[[plot_measure]]),
                  y = .data[[plot_measure]])) +
    geom_col(fill = "#51247a", width = .7) + coord_flip() +
    theme_bw(base_size = 12) +
    labs(title = title_str, x = "Collocate", y = y_label)
  print(p)
  save_excel(fcm_df, "collocations.xlsx")
  dir.create("MyOutput", showWarnings = FALSE, recursive = TRUE)
  ggsave("MyOutput/collocations_plot.png", bg="white", width=8, height=5, dpi=200)
  .ok("Saved <b>collocations.xlsx</b> and <b>collocations_plot.png</b> to MyOutput.")
}


<div style="background:#51247a;color:white;padding:10px 18px;border-radius:6px;margin:1.4rem 0 .4rem 0;font-family:sans-serif;"><span style="font-size:.75rem;opacity:.7;text-transform:uppercase;letter-spacing:.05em;">Step 4</span><br><b style="font-size:.98rem;">&#x2B07;&#xFE0F; Download your results</b></div>


<div style="background:#eafaf1;border-left:4px solid #27ae60;border-radius:6px;padding:12px 18px;margin:.6rem 0;font-family:sans-serif;font-size:.9rem;"><b style="color:#1a6b3a;">&#x2B07;&#xFE0F; Download your results</b><br>Files saved: <b>collocations.xlsx, collocations_plot.png</b><br>Open the <b>file browser</b> on the left, double-click <b>MyOutput</b>, then <b>right-click</b> any file and choose <b>Download</b>.</div>


---

### Citation

Schweinberger, Martin. (2025). *LADAL Collocation Analyser*. Brisbane: The University of Queensland. <https://ladal.edu.au/tools.html>

```bibtex
@misc{schweinberger2025collocations,
  author = {Schweinberger, Martin},
  title  = {LADAL Collocation Analyser},
  year   = {2025},
  organization = {The University of Queensland},
  url    = {https://ladal.edu.au/tools.html}
}
```


In [ ]:
# Uncomment to show package versions for reproducibility
# sessionInfo()
